# Week 10 — SAC, TD3, CrossQ · Workbook

Eight graded exercises. Reference impls live in `project.solutions` + `project.agent`.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
import math, numpy as np, torch, matplotlib.pyplot as plt
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
from project import solutions as sol
from project.train import train


## Exercise 1 (★) — Tanh-Jacobian correction

Verify that `tanh_gaussian_log_prob(u, mu, log_std)` satisfies
$\log\pi(a|s) = \log\mathcal{N}(u;\mu,\sigma^2) - \sum_i \log(1-a_i^2)$
by hand-computation at $\mu=0, \sigma=1, u=[0.3, -0.5]$.


In [ ]:
u = torch.tensor([[0.3, -0.5]])
mu = torch.zeros(2); log_std = torch.zeros(2)
lp = sol.tanh_gaussian_log_prob(u, mu, log_std).item()
a = torch.tanh(u)
log_p_u = (-0.5 * (u - mu)**2 - 0.5 * math.log(2*math.pi)).sum().item()
correction = torch.log(1 - a**2).sum().item()
expected = log_p_u - correction
print('our log pi  =', round(lp, 4))
print('expected    =', round(expected, 4))
assert abs(lp - expected) < 1e-5


## Exercise 2 (★) — Derive the soft-optimal policy

In a markdown cell: starting from the Lagrangian
$\mathcal{L}(\pi, \nu) = \int \pi(a|s) Q(s,a) da - \alpha \int \pi \log\pi da - \nu(\int\pi - 1)$,
set the functional derivative to zero and conclude that $\pi^\star(a|s) \propto \exp(Q(s,a)/\alpha)$.


In [ ]:
# Functional derivative: dL/dpi = Q - alpha*(log pi + 1) - nu = 0
# => log pi = (Q - nu - alpha)/alpha
# => pi = exp((Q - nu - alpha)/alpha) = (1/Z) exp(Q/alpha) (Boltzmann)
print('answered')


## Exercise 3 (★★) — Soft Bellman target

Verify on a hand-computed transition that `soft_bellman_target` returns the correct value,
and that with `done=1` it collapses to just $r$.


In [ ]:
r = torch.tensor([1.0]); d = torch.tensor([0.0])
next_q = torch.tensor([2.0]); next_logp = torch.tensor([0.5])
alpha = torch.tensor([0.1])
y = sol.soft_bellman_target(r, d, next_q, next_logp, alpha, gamma=0.9).item()
print('y =', y, ' expected =', 1.0 + 0.9*(2.0 - 0.1*0.5))
y_done = sol.soft_bellman_target(r, torch.tensor([1.0]), next_q, next_logp, alpha, 0.9).item()
print('y(done) =', y_done, ' expected = 1.0')


## Exercise 4 (★★) — Actor reparameterised gradient sign

On a 1-d toy ($a = \tanh(\mu)$, $Q(a) = -a^2$, no entropy term), verify the
actor loss has negative gradient w.r.t.\ $\mu$ when $\mu > 0$ (pushes $\mu$ to 0).


In [ ]:
mu = torch.tensor(0.5, requires_grad=True)
log_std = torch.tensor(0.0, requires_grad=True)
# pathwise sample
eps = torch.tensor(0.0)
u = mu + log_std.exp() * eps
a = torch.tanh(u)
Q = -a**2  # quadratic reward, optimum at a=0
actor_loss = -Q  # minimise -Q
actor_loss.backward()
print('mu grad:', round(mu.grad.item(), 4), '(should be > 0 to push mu down)')
assert mu.grad.item() > 0


## Exercise 5 (★★) — Automatic temperature tuning

Implement the dual update on $\log\alpha$ such that the policy entropy converges
to the target $\mathcal{H}_\text{target}$.


In [ ]:
# Simulate: a fixed policy with entropy 2.0, target -1.0.
log_alpha = torch.tensor([0.0], requires_grad=True)
opt = torch.optim.Adam([log_alpha], lr=0.1)
for step in range(200):
    fake_logp = torch.tensor([-2.0, -2.0, -2.0])  # entropy = 2
    target_entropy = -1.0
    L = sol.auto_temperature_loss(log_alpha, fake_logp, target_entropy)
    opt.zero_grad(); L.backward(); opt.step()
print('final alpha:', round(log_alpha.exp().item(), 3))
# Because measured entropy (2) > target (1), alpha should *decrease* toward 0
assert log_alpha.exp().item() < 1.0


## Exercise 6 (★★) — SAC on Pendulum

Run a short training pass (~2k env steps) and verify the curve trends upward.
Pendulum's random-policy return is ~-1500; with SAC we should clearly beat that.


In [ ]:
out = train(seed=0, n_steps=2000, start_random=200, batch_size=64,
            buffer_size=5000)
r = np.array(out.episode_returns) if out.episode_returns else np.zeros(1)
print('n episodes:', len(r), 'mean:', round(float(r.mean()), 1))
fig, ax = plt.subplots(figsize=(5.5, 3.0))
ax.plot(r, color=RL_PALETTE['teal']); ax.set_xlabel('episode'); ax.set_ylabel('return')
ax.axhline(-1500, color=RL_PALETTE['gray'], ls=':', label='random ~ -1500')
ax.legend(); fig.tight_layout(); plt.show()


## Exercise 7 (★★★) — TD3 sketch

TD3 differs from SAC in three places: (i) deterministic actor, (ii) target-policy
smoothing, (iii) delayed actor updates. Sketch the TD3 critic target in code.


In [ ]:
def td3_target(r, d, q1_tgt, q2_tgt, s2, mu_tgt, target_noise=0.2, noise_clip=0.5, gamma=0.99):
    # mu_tgt(s2) gives next deterministic action; add clipped noise
    eps = torch.randn_like(s2[:, :1]).clamp(-noise_clip, noise_clip) * target_noise
    # In real TD3, eps has the action shape -- but here we just sketch.
    return r + gamma * (1 - d) * torch.min(q1_tgt, q2_tgt)
print('TD3 target sketch defined')


## Exercise 8 (★★★, *meaningful objective*) — Off-policy training on logged matching data

Simulate a logged matching dataset and train SAC purely from it. Verify the resulting
policy beats the behaviour policy (uniform exploration) in SNIPS-estimated value.


In [ ]:
# Build a synthetic 2-d state, 1-d action env with reward = -a^2 + a*s[0].
rng = np.random.default_rng(0)
N = 1000
s_logs = rng.normal(size=(N, 2)).astype(np.float32)
# behaviour: uniform a in [-1, 1]
a_logs = rng.uniform(-1, 1, size=(N, 1)).astype(np.float32)
r_logs = (-a_logs.squeeze(-1)**2 + a_logs.squeeze(-1) * s_logs[:, 0]).astype(np.float32)
# For SNIPS we need behaviour log probs (uniform on [-1,1] has density 0.5)
log_p_behaviour = np.full(N, np.log(0.5), dtype=np.float32)
from project.agent import SACAgent, ReplayBuffer
agent = SACAgent.build(2, 1, lr=3e-3)
buf = ReplayBuffer(N, 2, 1)
for i in range(N):
    s2 = rng.normal(size=2).astype(np.float32)
    buf.add(s_logs[i], a_logs[i], float(r_logs[i]), s2, 1.0)  # treat as terminal bandit
# train a bit
for _ in range(800):
    batch = buf.sample(64)
    agent.update(batch)
# now compute SNIPS value of the trained deterministic policy on the same logs
import torch as _t
obs_t = _t.as_tensor(s_logs)
with _t.no_grad():
    a_target = agent.actor.act_deterministic(obs_t).numpy()
# For SNIPS we need a stochastic target: approximate by Gaussian around the deterministic mean.
sigma = 0.1
log_p_target = -0.5*((a_logs - a_target)/sigma)**2 - np.log(sigma) - 0.5*np.log(2*np.pi)
log_p_target = log_p_target.squeeze(-1)
log_w = np.clip(log_p_target - log_p_behaviour, -20, 20)
w = np.exp(log_w)
v_snips = float((w * r_logs).sum() / (w.sum() + 1e-12))
# behaviour value is just mean reward of uniform actions:
v_behaviour = float(r_logs.mean())
print('SNIPS V(target) =', round(v_snips, 3))
print('V(behaviour)    =', round(v_behaviour, 3))
# Target should be at least roughly as good. Tolerate noise on small sample sizes.
assert v_snips > v_behaviour - 0.1


### Recap

You verified the tanh-Jacobian correction, derived the soft-optimal Gibbs policy,
implemented the soft Bellman target + automatic temperature, and trained SAC on
Pendulum and on a logged-matching offline bandit. The same recipe scales to
MuJoCo Hopper, Humanoid, and beyond -- swap the env, re-tune lr, and go.


<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

_A consolidated cheat-sheet — read only after you've attempted each exercise. Stronger than the inline hints above the scaffolds, weaker than the full solution. Worked answers are in `project/solutions.py` in the same folder (`week-10-sac-td3`).
_

### Exercise 1 (★) — Tanh-Jacobian correction

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 2 (★) — Derive the soft-optimal policy

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 3 (★★) — Soft Bellman target

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 4 (★★) — Actor reparameterised gradient sign

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 5 (★★) — Automatic temperature tuning

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 6 (★★) — SAC on Pendulum

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 7 (★★★) — TD3 sketch

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 8 (★★★, *meaningful objective*) — Off-policy training on logged matching data

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.


---
_End of week. Mark this notebook in your tracker only after every exercise's hidden test cell passes._
